# 02. Análisis con Boxplots y Evolución de Versiones

**Objetivo:** Evaluar la evolución temporal de los flujos de trabajo GH-AW mediante boxplots en Seaborn, resúmenes estadísticos de cuartiles (IQR, Q1, Q3, Mediana), e inspeccionar casos atípicos en cambios de contenido y frontmatter.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

PROCESSED_DIR = "data/processed"

def calculate_boxplot_stats(df, group_col, value_col):
    """Calcula n, Mediana, Q1, Q3 e IQR para cada grupo."""
    def stats(series):
        s = series.dropna()
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        return pd.Series({
            "n_obs": len(s),
            "Mediana": s.median(),
            "Q1 (25%)": q1,
            "Q3 (75%)": q3,
            "IQR": iqr
        })
    return df.groupby(group_col)[value_col].apply(stats).unstack().reset_index()

## 1. Carga de Datos Procesados

In [ ]:
df_version_measures = pd.read_parquet(os.path.join(PROCESSED_DIR, "version_measures.parquet"))
df_trans = pd.read_parquet(os.path.join(PROCESSED_DIR, "transitions.parquet"))
file_summary = pd.read_parquet(os.path.join(PROCESSED_DIR, "file_summary.parquet"))
monthly_summary = pd.read_parquet(os.path.join(PROCESSED_DIR, "monthly_file_summary.parquet"))

print(f"Resúmenes por Archivo: {len(file_summary)}")
print(f"Transiciones Mensuales: {len(monthly_summary)}")

## 2. Comparación entre la Primera y la Última Versión

In [ ]:
# Filtrar historias con al menos 2 versiones
valid_files = file_summary[file_summary["version_count"] >= 2].dropna(subset=["initial_word_count", "final_word_count"]).copy()

# Melt para formatear datos en Seaborn
df_melt = valid_files.melt(
    id_vars=["source_markdown_file_history_id"],
    value_vars=["initial_word_count", "final_word_count"],
    var_name="Estado", value_name="Word_Count"
)
df_melt["Estado"] = df_melt["Estado"].map({"initial_word_count": "Primera Versión", "final_word_count": "Última Versión"})

# Tabla de Cuartiles
stats_comp = calculate_boxplot_stats(df_melt, "Estado", "Word_Count")
print("--- Tabla Resumen: Cuartiles e IQR ---")
display(stats_comp)

# Porcentajes de Cambio Neto individual
n_total = len(valid_files)
increased = (valid_files["net_body_change"] > 0).sum()
decreased = (valid_files["net_body_change"] < 0).sum()
equal = (valid_files["net_body_change"] == 0).sum()

pct_df = pd.DataFrame({
    "Comportamiento": ["Aumentó Longitud", "Disminuyó Longitud", "Permaneció Igual"],
    "Cantidad": [increased, decreased, equal],
    "Porcentaje (%)": [round(increased/n_total*100, 2), round(decreased/n_total*100, 2), round(equal/n_total*100, 2)]
})
display(pct_df)

# Gráfico Boxplot
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_melt, x="Estado", y="Word_Count", palette="Set2", ax=ax, whis=1.5)
sns.stripplot(data=df_melt, x="Estado", y="Word_Count", color="black", alpha=0.3, jitter=0.2, ax=ax)
ax.set_title(f"Comparación de Longitud del Body (Primera vs Última Versión) [n={n_total}]")
ax.set_ylabel("Palabras en el Body")
plt.tight_layout()
plt.show()

## 3. Variación de la Magnitud de Cambio de Longitud por Mes

In [ ]:
# Ordenar meses cronológicamente
monthly_summary = monthly_summary.sort_values("year_month")

# Tabla de Cuartiles por Mes
stats_month = calculate_boxplot_stats(monthly_summary, "year_month", "median_abs_body_change")
print("--- Tabla Resumen Mensual de Cuartiles ---")
display(stats_month)

# Gráfico
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=monthly_summary, x="year_month", y="median_abs_body_change", palette="Blues", ax=ax, whis=1.5)
sns.stripplot(data=monthly_summary, x="year_month", y="median_abs_body_change", color="red", alpha=0.4, jitter=0.2, ax=ax)
ax.set_title("Variación de la Magnitud de Cambio de Longitud del Body por Mes")
ax.set_xlabel("Año-Mes")
ax.set_ylabel("Mediana del Cambio Absoluto (Palabras)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Tiempo entre Versiones Consecutivas

In [ ]:
valid_times = file_summary.dropna(subset=["median_time_between_versions_days"]).copy()
valid_times["Grupo"] = "Tiempo entre Versiones"

stats_time = calculate_boxplot_stats(valid_times, "Grupo", "median_time_between_versions_days")
print("--- Estadísticos de Tiempo entre Versiones (Días) ---")
display(stats_time)

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=valid_times, x="Grupo", y="median_time_between_versions_days", color="lightcoral", whis=1.5, ax=ax)
sns.stripplot(data=valid_times, x="Grupo", y="median_time_between_versions_days", color="black", alpha=0.3, ax=ax)
ax.set_title(f"Distribución de la Mediana del Tiempo entre Versiones (n={len(valid_times)})")
ax.set_ylabel("Días")
plt.tight_layout()
plt.show()

## 5. Inspección de Casos de Cambio Seleccionados

Seleccionamos dos transiciones clave para inspección cualitativa:
1. **Caso 1**: Transición con una magnitud de cambio de longitud del body atípicamente alta.
2. **Caso 2**: Transición con modificación en la cantidad de claves de primer nivel del Frontmatter.

In [ ]:
# Caso 1: Mayor cambio absoluto en body
case1 = df_trans.nlargest(1, "abs_body_length_change").iloc[0]

# Caso 2: Cambio en claves de frontmatter
case2_candidates = df_trans[df_trans["frontmatter_keys_change"] != 0]
if len(case2_candidates) > 0:
    case2 = case2_candidates.iloc[0]
else:
    case2 = df_trans.nlargest(2, "abs_body_length_change").iloc[1]

cases_df = pd.DataFrame([
    {
        "Tipo": "Caso 1 (Mayor Cambio de Body)",
        "Version Curr ID": case1["source_markdown_file_version_id_curr"],
        "Version Prev ID": case1["source_markdown_file_version_id_prev"],
        "Fecha Curr": case1["committed_at_curr"],
        "Cambio Palabras": case1["body_length_change"],
        "Cambio Claves FM": case1["frontmatter_keys_change"]
    },
    {
        "Tipo": "Caso 2 (Cambio en Frontmatter)",
        "Version Curr ID": case2["source_markdown_file_version_id_curr"],
        "Version Prev ID": case2["source_markdown_file_version_id_prev"],
        "Fecha Curr": case2["committed_at_curr"],
        "Cambio Palabras": case2["body_length_change"],
        "Cambio Claves FM": case2["frontmatter_keys_change"]
    }
])

display(cases_df)

## 6. Hallazgos y Limitaciones

### Hallazgos Principales:
1. **Inercia en el Body vs Revisiones Puntuales**: La mayoría de las transiciones consecutivas muestran un cambio de longitud cercano a cero palabras, concentrándose la expansión del contenido en hitos o versiones puntuales.
2. **Evolución Asimétrica del Frontmatter**: Los cambios en las claves de primer nivel del YAML son poco frecuentes tras la creación del archivo, lo que indica que el esquema o contrato del agente se estabiliza de forma temprana.
3. **Frecuencia de Actualización Asincrónica**: El tiempo entre versiones presenta un comportamiento con sesgo positivo (derecha), donde la mediana de actualización oscila en pocos días, pero existen outliers significativos que abarcan varios meses.

### Limitaciones:
* **Cobertura del Dataset**: El análisis está supeditado a los commits observados en el dataset GHAW-H.
* **Sustituciones de Texto**: Un cambio de 0 palabras en el body no descarta la sustitución de fragmentos o refactorización de instrucciones en lenguaje natural.
* **Formato YAML Heterogéneo**: Errores de sintaxis en el Frontmatter impiden contabilizar las claves en ciertas versiones intermedias.